# CUDA Particle Simulation — Colab Demo

Real-time GPU particle simulation running on a Tesla T4.  
100 particles on crossing H/V lines — elastic wall bounce + particle collision.  

**Runtime:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Verify T4 GPU is attached
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
import os

REPO = '/content/cuda_particles'
if not os.path.exists(REPO):
    !git clone https://github.com/GaiBrutman/cuda_particles.git {REPO}
else:
    !git -C {REPO} pull --ff-only

%cd {REPO}

In [ ]:
# Build for T4 (SM 7.5 = Turing)
BUILD = f'{REPO}/build'
!cmake {REPO} -B {BUILD} \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=75 \
    -DCPU_ONLY=OFF \
    -DPROFILING={str(PROFILING)}
!cmake --build {BUILD} --parallel $(nproc)

In [ ]:
import subprocess, pathlib

FRAMES = '/tmp/frames'
pathlib.Path(FRAMES).mkdir(exist_ok=True)

result = subprocess.run([
    f'{BUILD}/particles',
    '--particles', '100',
    '--frames',    '300',
    '--width',     '1280',
    '--height',    '720',
    '--dt',        '0.016',
    '--output',    FRAMES,
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

frames = sorted(pathlib.Path(FRAMES).glob('frame_*.png'))
print(f'Generated {len(frames)} frames')

In [ ]:
if not PROFILING:
    print('Profiling disabled — rebuild with PROFILING=True in the build cell.')
else:
    # ── Profiling: parse timing output and plot phase breakdown ──────────────────
    import re
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    
    # Parse per-frame timing lines printed by the binary:
    # "<frame>  <sim>  <render>  <readback>  <save>"
    frame_re = re.compile(
        r'^(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)',
        re.MULTILINE
    )
    
    rows = frame_re.findall(result.stdout)
    if not rows:
        print('No timing data found — re-run the simulation cell first.')
    else:
        frames_idx = [int(r[0])   for r in rows]
        sim        = [float(r[1]) for r in rows]
        render     = [float(r[2]) for r in rows]
        readback   = [float(r[3]) for r in rows]
        save       = [float(r[4]) for r in rows]
        total      = [sim[i]+render[i]+readback[i]+save[i] for i in range(len(rows))]
    
        phases = ['sim', 'render', 'readback', 'save']
        avgs   = [np.mean(sim), np.mean(render), np.mean(readback), np.mean(save)]
        colors = ['#4c8cbf', '#e0883a', '#59a869', '#d9534f']
    
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
        fig.suptitle('Frame timing breakdown', fontsize=12, fontweight='bold')
    
        # ── Left: avg ms per phase ────────────────────────────────────────────────
        bars = ax1.bar(phases, avgs, color=colors, width=0.5, edgecolor='white')
        ax1.set_ylabel('avg ms / frame')
        ax1.set_title('Phase averages')
        for bar, v in zip(bars, avgs):
            ax1.text(bar.get_x() + bar.get_width()/2, v + max(avgs)*0.01,
                     f'{v:.3f}', ha='center', va='bottom', fontsize=9)
        total_avg = sum(avgs)
        ax1.set_ylim(0, total_avg * 1.15)
        ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
    
        # ── Right: stacked frame time over run ────────────────────────────────────
        ax2.stackplot(frames_idx,
                      sim, render, readback, save,
                      labels=phases, colors=colors, alpha=0.85)
        ax2.set_xlabel('frame')
        ax2.set_ylabel('ms')
        ax2.set_title('Stacked frame time')
        ax2.legend(loc='upper right', fontsize=8)
    
        plt.tight_layout()
        plt.show()
    
        # ── Summary table ─────────────────────────────────────────────────────────
        print(f'\n{"phase":<12} {"avg ms":>8} {"% of frame":>10}')
        print('-' * 32)
        for p, a in zip(phases, avgs):
            print(f'{p:<12} {a:>8.3f} {a/total_avg*100:>9.1f}%')
        print('-' * 32)
        print(f'{"total":<12} {total_avg:>8.3f}')
        fps = 1000.0 / total_avg if total_avg > 0 else 0
        print(f'\nbottleneck: {phases[avgs.index(max(avgs))]}  '
              f'({max(avgs):.3f} ms, {max(avgs)/total_avg*100:.1f}% of frame)')
        print(f'effective fps (avg frame time): {fps:.1f}')
        print('\nnote: in CUDA mode, "readback" includes GPU compute + D2H transfer;'
              ' sim/render show only async kernel-launch latency.')


In [ ]:
# ── Scale test: 500 K particles ───────────────────────────────────────────────
# Uncomment to benchmark large particle count on T4.

# import time
# start = time.perf_counter()
# subprocess.run([
#     f'{BUILD}/particles',
#     '--particles', '500000',
#     '--frames',    '60',
#     '--output',    '/tmp/frames_large',
# ], check=True)
# elapsed = time.perf_counter() - start
# print(f'500K × 60 frames in {elapsed:.1f}s  ({60/elapsed:.1f} sim-fps)')

In [ ]:
!pip install -q matplotlib pillow

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from PIL import Image
from IPython.display import HTML

paths  = sorted(glob.glob(f'{FRAMES}/frame_*.png'))
frames_arr = [np.asarray(Image.open(p).convert('RGBA')) for p in paths]

h, w = frames_arr[0].shape[:2]
fig, ax = plt.subplots(figsize=(w / 100, h / 100), dpi=100)
fig.patch.set_facecolor('black')
ax.axis('off')
fig.tight_layout(pad=0)

im      = ax.imshow(frames_arr[0], origin='upper', aspect='equal',
                    extent=[0, w, h, 0], animated=True)
counter = ax.text(8, 14, 'frame 0000', color='gray',
                  fontsize=8, fontfamily='monospace', va='top')

def update(i):
    im.set_data(frames_arr[i])
    counter.set_text(f'frame {i:04d}')
    return im, counter

anim = animation.FuncAnimation(fig, update, frames=len(frames_arr),
                                interval=33, blit=True)
HTML(anim.to_jshtml())

In [ ]:
# Export as GIF (optional — takes ~30s for 300 frames)
GIF = '/tmp/particles.gif'
writer = animation.PillowWriter(fps=30)
anim.save(GIF, writer=writer)
print(f'Saved {GIF}')

from IPython.display import Image as IPyImage
IPyImage(GIF)